<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def beam_splitter_qubit(r):
    """
    Returns the correct 4x4 beam splitter matrix
    acting on the single-photon subspace encoded in qubits.

    Args:
        r (float): The reflection coefficient of the beam splitter.

    Returns:
        (np.ndarray): 4 x 4 unitary matrix for the beam splitter.
    """
    t = np.sqrt(1 - r**2)
    # Basis: |00>, |01>, |10>, |11>
    # |00>: 0, |01>: 1, |10>: 2, |11>: 3
    U = np.zeros((4,4), dtype=complex)
    U[0,0] = 1.0
    U[3,3] = 1.0
    U[1,1] = r
    U[1,2] = t
    U[2,1] = t
    U[2,2] = -r
    return U

dev = qml.device('default.qubit', wires=2)

def mz_interferometer(r):
    """
    Returns probabilities p_AC and p_D as described in the challenge.

    Args:
        r (float): Reflection coefficient.

    Returns:
        np.array: [p_AC, p_D]
    """
    # --- First branch: Probability that A triggers (after first beam splitter)
    @qml.qnode(dev)
    def first_bs_probs(r):
        # Start in |01>: wire 1=0, wire 0=1 (rightmost is wire 0)
        qml.BasisState(np.array([0, 1]), wires=[1,0])
        qml.QubitUnitary(beam_splitter_qubit(r), wires=[1,0])
        return qml.probs(wires=[1,0])

    probs_after_bs1 = first_bs_probs(r)
    # Resulting state order: |00> (0), |01> (1), |10> (2), |11> (3)
    p_A = probs_after_bs1[1]   # |01>: photon at wire 0 (detected by A)
    p_notA = probs_after_bs1[2]  # |10>: photon at wire 1 (NOT A, continue interferometer)

    # --- Second branch: Postselect on |10>, then second beam splitter, and get probabilities
    @qml.qnode(dev)
    def second_bs_probs(r):
        # Start in |10>: wire 1=1, wire 0=0
        qml.BasisState(np.array([1, 0]), wires=[1,0])
        qml.QubitUnitary(beam_splitter_qubit(r), wires=[1,0])
        return qml.probs(wires=[1,0])

    if p_notA > 1e-12:
        probs_after_bs2 = second_bs_probs(r)
        # |01>: index 1 => C, |10>: index 2 => D
        p_C = p_notA * probs_after_bs2[1]
        p_D = p_notA * probs_after_bs2[2]
    else:
        p_C = 0.0
        p_D = 0.0

    p_AC = float(p_A + p_C)
    p_D = float(p_D)
    return np.array([p_AC, p_D])

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = mz_interferometer(float(ins)).tolist()
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol=1e-4), "Not the correct probabilities"

# === Test cases ===
def test_mz():
    test_cases = [
        ('0.5', '[0.8125, 0.1875]'),
        ('0.577350269', '[0.777778, 0.222222]')
    ]
    for i, (input_, expected_output) in enumerate(test_cases):
        print(f"Running test case {i} with input '{input_}'...")
        output = run(input_)
        try:
            check(output, expected_output)
        except Exception as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
            raise
        else:
            print("Correct!")

if __name__ == "__main__":
    test_mz()


Running test case 0 with input '0.5'...


TypeError: 'StateMP' object is not subscriptable